# Download and Prepare CAMELS-GB Dataset

This notebook automates the process of downloading the CAMELS-GB dataset and extracting it to the correct location within the project structure.

In [ ]:
import requests
import zipfile
import os
from io import BytesIO
from tqdm import tqdm

## Configuration

In [ ]:
DATASET_URL = "https://catalogue.ceh.ac.uk/download/8344e4f3-d2ea-44f5-8afa-86d2987543a9?url=https%3A%2F%2Fdata-package.ceh.ac.uk%2Fdata%2F8344e4f3-d2ea-44f5-8afa-86d2987543a9.zip"
EXTRACT_PATH = "../datasets/camels-gb/"

## Download the Dataset

In [ ]:
print("Downloading dataset... This may take a few moments.")
response = requests.get(DATASET_URL, stream=True)

if response.status_code == 200:
    total_size = int(response.headers.get('content-length', 0))
    block_size = 1024 # 1 Kibibyte
    
    zip_content = BytesIO()
    with tqdm(total=total_size, unit='iB', unit_scale=True) as progress_bar:
        for data in response.iter_content(block_size):
            progress_bar.update(len(data))
            zip_content.write(data)
    print("Download complete.")
else:
    print(f"Error downloading file: Status code {response.status_code}")

## Extract the Dataset

We will now extract the contents of the `data` directory from the downloaded zip file into `datasets/camels-gb/`.

In [ ]:
if response.status_code == 200:
    with zipfile.ZipFile(zip_content) as z:
        # Create a list of all files in the zip
        file_list = z.namelist()
        
        # Filter for files within the 'data/' directory of the zip file
        data_files = [f for f in file_list if f.startswith('data/')]
        
        # Extract only those files
        print(f"Extracting {len(data_files)} files to {os.path.abspath(EXTRACT_PATH)}...")
        for file in tqdm(data_files):
            z.extract(file, path=EXTRACT_PATH)
        print("Extraction complete.")
        
        # Now, let's handle the nested zip file for catchment boundaries
        boundaries_zip_path_in_zip = 'data/CAMELS_GB_catchment_boundaries.zip'
        if boundaries_zip_path_in_zip in file_list:
            print("Extracting catchment boundaries...")
            with z.open(boundaries_zip_path_in_zip) as nested_z_file:
                with zipfile.ZipFile(BytesIO(nested_z_file.read())) as nested_z:
                    nested_z.extractall(os.path.join(EXTRACT_PATH, 'data', 'CAMELS_GB_catchment_boundaries'))
            print("Catchment boundaries extracted.")
            # Clean up the nested zip file
            os.remove(os.path.join(EXTRACT_PATH, 'data', 'CAMELS_GB_catchment_boundaries.zip'))

## Final Check

Let's verify that the files have been extracted to the correct location.

In [ ]:
final_path = os.path.join(EXTRACT_PATH, 'data')
if os.path.exists(final_path):
    print(f"Successfully extracted files to: {os.path.abspath(final_path)}")
    print("Contents:")
    for item in os.listdir(final_path)[:10]: # Print first 10 items
        print(f"- {item}")
else:
    print("Extraction seems to have failed. Please check the paths and permissions.")